## Language Model Evaluation
#### Evaluation of the three trained transformer models on multilingual test data 

### 1. Environment and Model Loading

In [1]:
# Import libraries for model evaluation
import torch # PyTorch machine-learning framework
import torch.nn as nn # neural network components
import torch.nn.functional as F # neural network mathematical operations
from torch.utils.data import Dataset, DataLoader # tools for organizing datasets and batches
from torch.nn.utils.rnn import pad_sequence # pad shorter sequences for batching

import sentencepiece as spm # load and use trained BPE tokenizers

import json # read and write JSON files

import math # mathematical functions

from pathlib import Path # filesystem path handling

from collections import Counter # tool for counting occurrences of individual items

# Define project and data directories
PROJECT_DIR = Path.cwd().parent # identify project directory from notebook location
DATA_DIR = Path("/srv/data/lt2326-h26/a1") # location of assignment datasets

TOKENIZER_DIR = PROJECT_DIR / "models" / "tokenizers" # location of saved tokenizers
MODEL_DIR = PROJECT_DIR / "models" / "language_models" # location of trained language models

# Select GPU if available
device = torch.device( # select computing device
    "cuda" if torch.cuda.is_available() else "cpu" # use GPU if available; otherwise CPU
)

print("Project directory:", PROJECT_DIR) # display project directory
print("Evaluation device:", device) # display selected computing device

Project directory: /home/gussougr@GU.GU.SE/lt2326-assignment1
Evaluation device: cuda


In [2]:
# Shared Transformer architecture configuration
MODEL_CONFIG = { # dictionary of transformer hyperparameters
    "d_model": 256, # 256 dimensions per vector (token representation)
    "n_heads": 4, # 4 attention heads
    "n_layers": 2, # 2 transformer layers
    "d_ff": 1024, # 1024 feed forward dimension
    "dropout": 0.10, # 10% dropout
    "context_length": 128, # maximum sequence length of 128 tokens
}

print("Transformer configuration:") # print label
for parameter, value in MODEL_CONFIG.items(): # iterate over keys and values in MODEL_CONFIG dictionary
    print(f"{parameter}: {value}") # for each entry print label and value

Transformer configuration:
d_model: 256
n_heads: 4
n_layers: 2
d_ff: 1024
dropout: 0.1
context_length: 128


In [3]:
# Define the Transformer language model
class TransformerLM(nn.Module): # create transformer/neural network class

    def __init__(self, vocab_size, config): # initialize the model with parameters including vocabulary size and hyperparameters
        super().__init__() # initialize neural network from Pytorch

        d_model = config["d_model"] # set dimensions for token representation (retrieved from above dictionary)
        context_length = config["context_length"] # set sequence length (retrieved from above dictionary)

        self.token_embedding = nn.Embedding(vocab_size, d_model) # embed tokens from input token IDs with vocabulary model size and dimensions set above

        # Learn a vector representation for each position
        self.position_embedding = nn.Embedding(context_length, d_model) # positional embeddings using sequence length and dimensions set above

        # Create one Transformer layer
        encoder_layer = nn.TransformerEncoderLayer( # create transformer layer via Pytorch; hyperparameters below
            d_model=d_model, # set dimensions for token representation (retrieved from above dictionary)
            nhead=config["n_heads"], # set number of attention heads (retrieved from above dictionary)
            dim_feedforward=config["d_ff"], # set feed forward dimension (retrieved from above dictionary)
            dropout=config["dropout"], # set dropout (retrieved from above dictionary)
            activation="gelu", # nonlinear activation (GELU)
            batch_first=True, # input data organization (batch, sequence, features)
            norm_first=True # pre-layer normalization (prior to attention/feed-forward)
        )

        # Stack two Transformer layers
        self.transformer = nn.TransformerEncoder( # create hidden layers via Pytorch; hyperparameters below
            encoder_layer, # input from transformer layer
            num_layers=config["n_layers"], # set transformer layers (retrieved from above dictionary)
            enable_nested_tensor=False # variable length sequence optimization disabled
        )

        self.final_norm = nn.LayerNorm(d_model) # normalize token representations utilizing determined dimensions (retrieved from above dictionary)

        # Convert token representations into vocabulary prediction scores
        self.output_layer = nn.Linear(d_model, vocab_size) # convert dimensional representations to prediction scores

    # Define how input data moves through the model
    def forward(self, input_ids): # define action of transformer using token IDs

        # Get the dimensions of the input
        batch_size, seq_len = input_ids.shape # shape of input: (batch, sequence length)

        # Create position IDs for each token
        positions = torch.arange( # create position numbers
            seq_len, # sequence length
            device=input_ids.device # use same device as input data
        )

        # Look up token and positional embeddings
        token_embeddings = self.token_embedding(input_ids) # retreive token embeddings of input data
        position_embeddings = self.position_embedding(positions) # retrieve position embeddings from above 

        # Combine token information with position information
        x = token_embeddings + position_embeddings # combine token and positional embeddings

        # Create a causal attention mask
        causal_mask = torch.triu( # keep upper triangular portion of grid (triu = 'triangular upper')
            torch.ones( # create grid of ones
                seq_len, # first dimension of grid
                seq_len, # second dimension of grid
                device=input_ids.device, # use same device as input data
                dtype=torch.bool # create boolean values (boolean mask: false = attention permitted, true = attention blocked)
            ),
            diagonal=1 # start upper triangle 1 position above main diagonal
        )

        # Process embeddings through Transformer layers using causal attention
        x = self.transformer(x, mask=causal_mask) # input combined token and positional embeddings ('x') through transformer; enable causal mask
        x = self.final_norm(x) # final layer normalization (scale adjustment) of updated representations
        logits = self.output_layer(x) # convert combined representations into vocabulary prediction scores

        return logits # return prediction scores

In [4]:
# Read vocabulary sizes from saved tokenizer files
with open(TOKENIZER_DIR / "character_vocab.json", "r", encoding="utf-8") as f:
    saved_char_vocab = json.load(f)

vocab_sizes = {
    "character": len(saved_char_vocab["char_to_id"]) + 1,
    "bpe_2000": spm.SentencePieceProcessor(
        model_file=str(TOKENIZER_DIR / "bpe_2000.model")
    ).get_piece_size(),
    "bpe_10000": spm.SentencePieceProcessor(
        model_file=str(TOKENIZER_DIR / "bpe_10000.model")
    ).get_piece_size()
}

print("Vocabulary sizes:", vocab_sizes)

models = {} # create dictionary to store loaded Transformer model objects

for name, vocab_size in vocab_sizes.items(): # iterate over tokenizer names and vocabulary sizes

    # Create model with the original architecture
    model = TransformerLM( # create Transformer model object using original architecture
        vocab_size=vocab_size, # use current tokenizer's vocabulary size
        config=MODEL_CONFIG # use original model configuration
    ).to(device) # move model to selected computing device

    # Load saved weights
    weights_path = MODEL_DIR / f"{name}_weights.pt" # define filepath for saved model weights

    state_dict = torch.load( # load saved model parameters
        weights_path, # location of current model's weights file
        map_location=device, # load tensors onto selected computing device
        weights_only=True # restrict loading to supported safe data types
    )

    model.load_state_dict(state_dict) # insert saved trained weights into model

    # Switch to evaluation mode (disables dropout)
    model.eval() # set model to evaluation mode

    models[name] = model # store loaded model object under its tokenizer name

    print(f"Loaded {name} successfully", flush=True) # confirm successful model loading

print("\nAll three trained models ready for evaluation.") # confirm all models loaded

Vocabulary sizes: {'character': 9443, 'bpe_2000': 2000, 'bpe_10000': 10000}
Loaded character successfully
Loaded bpe_2000 successfully
Loaded bpe_10000 successfully

All three trained models ready for evaluation.


### 2. Test Data Preparation

In [5]:
# Load saved character vocabulary
with open(TOKENIZER_DIR / "character_vocab.json", "r", encoding="utf-8") as f: # open saved character vocabulary file
    char_vocab = json.load(f) # load character vocabulary dictionary from JSON

char_to_id = char_vocab["char_to_id"] # retrieve character-to-ID mapping
char_unk_id = char_vocab["unk_id"] # retrieve unknown character ID

# Load saved BPE tokenizers
bpe_small = spm.SentencePieceProcessor( # load saved small BPE tokenizer
    model_file=str(TOKENIZER_DIR / "bpe_2000.model") # specify small tokenizer model file
)

bpe_large = spm.SentencePieceProcessor( # load saved large BPE tokenizer
    model_file=str(TOKENIZER_DIR / "bpe_10000.model") # specify large tokenizer model file
)

# Define test files
test_files = { # create dictionary containing test data filepaths
    "en": DATA_DIR / "test" / "en.txt", # English test data
    "tr": DATA_DIR / "test" / "tr.txt", # Turkish test data
    "zh": DATA_DIR / "test" / "zh.txt" # Chinese test data
}

# Load test sentences
test_texts = {} # create dictionary to hold test sentences by language

for lang, path in test_files.items(): # iterate over languages and their test filepaths
    with open(path, "r", encoding="utf-8") as f: # open current language's test file
        test_texts[lang] = [ # store current language's test sentences
            line.strip() # remove leading/trailing whitespace and newline characters
            for line in f # iterate over lines in test file
            if line.strip() # exclude empty or whitespace-only lines
        ]

    print(f"{lang}: {len(test_texts[lang]):,} test sentences") # display test sentence count

en: 4,472 test sentences
tr: 3,792 test sentences
zh: 9,430 test sentences


In [6]:
# Convert sentences into token IDs using the selected tokenizer
def encode_sentences(texts, tokenizer_type): # define function to encode sentences using specified tokenizer

    encoded = {} # create dictionary to store encoded sentences by language

    for lang, sentences in texts.items(): # iterate over languages and their sentences

        if tokenizer_type == "character": # use character-level tokenizer
            encoded[lang] = [ # store encoded sentences for current language
                [char_to_id.get(char, char_unk_id) for char in sentence] # convert characters to IDs, using unknown ID as fallback
                for sentence in sentences # iterate over sentences
            ]

        elif tokenizer_type == "bpe_2000": # use small BPE tokenizer
            encoded[lang] = [
                bpe_small.encode(sentence, out_type=int) # convert sentence into integer token IDs
                for sentence in sentences
            ]

        elif tokenizer_type == "bpe_10000": # use large BPE tokenizer
            encoded[lang] = [
                bpe_large.encode(sentence, out_type=int) # convert sentence into integer token IDs
                for sentence in sentences
            ]

        else:
            raise ValueError(f"Unknown tokenizer: {tokenizer_type}") # raise error for unrecognized tokenizer

    return encoded # return dictionary of encoded sentences

In [7]:
class LanguageModelDataset(Dataset): # LanguageModelDataset class definition

    def __init__(self, encoded_sentences, context_length=128): # initializer method with encoded sentences and context length (default 128)

        self.examples = [] # create a list belonging to LanguageModelDataset to hold tuples of input/target tensors per sequence

        for lang, sentences in encoded_sentences.items(): # iterate over dataset languages

            for tokens in sentences: # iterate over encoded sentences

                # Need at least two tokens for next-token prediction
                if len(tokens) < 2: # check whether sentence contains fewer than two tokens
                    continue # skip sentence and move to next iteration

                # Create consecutive chunks with shifted targets
                for start in range(0, len(tokens) - 1, context_length): # iterate through token positions in steps of 128, stopping before final starting position

                    y = tokens[start + 1:start + context_length + 1] # select target token IDs, shifted forward one position
                    x = tokens[start:start + context_length][:len(y)] # select input token IDs, matching target sequence length

                    self.examples.append(( # add tuple containing input and target tensors to examples list
                        torch.tensor(x, dtype=torch.long), # convert input token IDs into PyTorch tensor
                        torch.tensor(y, dtype=torch.long) # convert target token IDs into PyTorch tensor
                    ))

    def __len__(self): # method returning dataset length
        return len(self.examples) # return number of examples in dataset

    def __getitem__(self, index): # method retrieving dataset example by index
        return self.examples[index] # return selected input/target tensor tuple

In [8]:
def collate_lm_batch(batch): # definition of padding function for tensor batching

    inputs, targets = zip(*batch) # separate each input tuple into groups of inputs and targets

    # Pad input sequences with token ID 0
    padded_inputs = pad_sequence( # add padding value to shorter sequences in input group
        inputs,
        batch_first=True,
        padding_value=0 # add 0 as padding value
    )

    # Pad target sequences with -100 (ignored by loss function)
    padded_targets = pad_sequence( # add padding value to shorter sequences in target group
        targets,
        batch_first=True,
        padding_value=-100 # add -100 as padding value
    )

    return padded_inputs, padded_targets # return padded groups

### 3. Test-Set Evaluation

In [9]:
# Evaluate models on each language's test data
test_results = {} # create dictionary to store test results for each model and language

for name, model in models.items(): # iterate over the three trained Transformer models

    print(f"\nEvaluating {name}...", flush=True) # display current model being evaluated

    test_results[name] = {} # create dictionary for current model's language results

    for lang in ["en", "tr", "zh"]: # evaluate current model separately on each language

        # Encode this language's test sentences
        encoded = encode_sentences( # convert test sentences into token IDs
            {lang: test_texts[lang]}, # select current language's test sentences
            name # specify current model's tokenizer
        )

        # Create test dataset using training sequence construction
        dataset = LanguageModelDataset( # create test dataset object
            encoded, # use encoded test sentences
            context_length=MODEL_CONFIG["context_length"] # use original model context length
        )

        loader = DataLoader( # create DataLoader for test examples
            dataset, # use prepared test dataset
            batch_size=32, # process 32 examples per batch
            shuffle=False, # preserve original example order
            collate_fn=collate_lm_batch # use established tensor padding function
        )

        total_loss = 0.0 # initialize accumulated test loss
        total_tokens = 0 # initialize count of predicted test tokens

        model.eval() # set model to evaluation mode

        with torch.no_grad(): # disable gradient calculations during testing

            for inputs, targets in loader: # iterate over test batches

                inputs = inputs.to(device) # move input tensors to selected device
                targets = targets.to(device) # move target tensors to selected device

                logits = model(inputs) # generate prediction scores for test inputs

                loss = F.cross_entropy( # calculate test prediction error
                    logits.reshape(-1, logits.size(-1)), # reshape prediction scores for loss calculation
                    targets.reshape(-1), # flatten target token IDs
                    ignore_index=-100, # exclude padded target positions
                    reduction="sum" # sum prediction losses rather than averaging
                )

                total_loss += loss.item() # accumulate total prediction loss
                total_tokens += (targets != -100).sum().item() # count predicted tokens excluding padding

        # Original Unicode character count
        total_chars = sum( # count characters in original test sentences
            len(sentence) for sentence in test_texts[lang] # sum sentence lengths for current language
        )

        # Convert total negative log-likelihood to bits per character
        bpc = total_loss / (math.log(2) * total_chars) # convert accumulated loss to bits per original character

        test_results[name][lang] = { # store evaluation statistics for current model and language
            "bpc": bpc, # bits per character
            "total_loss": total_loss, # accumulated negative log-likelihood
            "total_tokens": total_tokens, # number of predicted tokens
            "total_characters": total_chars # number of original Unicode characters
        }

        print( # display current language's evaluation results
            f"  {lang}: BPC = {bpc:.4f} "
            f"({total_tokens:,} predicted tokens)",
            flush=True
        )

# Save results
with open(MODEL_DIR / "test_bpc_results.json", "w") as f: # open JSON results file for writing
    json.dump(test_results, f, indent=2) # save multilingual test results in readable JSON format

print("\nTest evaluation complete.") # confirm evaluation has finished


Evaluating character...
  en: BPC = 2.4213 (363,837 predicted tokens)
  tr: BPC = 2.4564 (364,550 predicted tokens)
  zh: BPC = 6.5608 (358,835 predicted tokens)

Evaluating bpe_2000...
  en: BPC = 2.2274 (197,022 predicted tokens)
  tr: BPC = 2.3275 (212,593 predicted tokens)
  zh: BPC = 6.2035 (343,235 predicted tokens)

Evaluating bpe_10000...
  en: BPC = 2.0405 (106,042 predicted tokens)
  tr: BPC = 2.1187 (105,517 predicted tokens)
  zh: BPC = 6.0474 (270,467 predicted tokens)

Test evaluation complete.


In [10]:
# Calculate overall BPC across all three languages
overall_results = {} # create dictionary to store overall BPC for each model

for name, language_results in test_results.items(): # iterate over models and their language-specific results

    total_loss = sum( # calculate combined prediction loss across all three languages
        result["total_loss"] # retrieve total loss for current language
        for result in language_results.values() # iterate over language results
    )

    total_chars = sum( # calculate combined character count across all three languages
        result["total_characters"] # retrieve character count for current language
        for result in language_results.values() # iterate over language results
    )

    overall_bpc = total_loss / (math.log(2) * total_chars) # calculate overall bits per character

    overall_results[name] = overall_bpc # store overall BPC under current model name

    print(f"{name}: Overall BPC = {overall_bpc:.4f}") # display overall BPC result

# Save overall results
with open(MODEL_DIR / "overall_bpc_results.json", "w") as f: # open JSON results file for writing
    json.dump(overall_results, f, indent=2) # save overall BPC results in readable JSON format

character: Overall BPC = 3.8127
bpe_2000: Overall BPC = 3.5860
bpe_10000: Overall BPC = 3.4020


### 4. Vocabulary Allocation Analysis

In [11]:
# Count BPE token occurrences by language in training data
def count_bpe_usage(tokenizer): # define function to count token usage for selected BPE tokenizer

    counts = {} # dictionary to store token frequency counts by language
    totals = {} # dictionary to store total token occurrences by language

    for lang in ["en", "tr", "zh"]: # iterate over three languages

        path = DATA_DIR / "train" / f"{lang}.txt" # identify current language's training file
        counts[lang] = Counter() # initialize token frequency counter for current language
        totals[lang] = 0 # initialize total token count for current language

        with open(path, "r", encoding="utf-8") as f: # open training file in read mode
            for line in f: # iterate over sentences in training file

                sentence = line.strip() # remove leading/trailing whitespace

                if not sentence: # check for empty sentence
                    continue # skip empty sentence

                token_ids = tokenizer.encode( # encode sentence using selected BPE tokenizer
                    sentence,
                    out_type=int # return token IDs as integers
                )

                counts[lang].update(token_ids) # add token occurrences to language frequency counter
                totals[lang] += len(token_ids) # accumulate total number of tokens for language

    return counts, totals # return token frequency dictionaries and total token counts


# Count usage for both main BPE tokenizers
bpe_usage = {} # create dictionary to store token usage results for both BPE tokenizers

for name, tokenizer in [ # iterate over tokenizer names and corresponding tokenizer objects
    ("bpe_2000", bpe_small),
    ("bpe_10000", bpe_large)
]:

    counts, totals = count_bpe_usage(tokenizer) # calculate token frequencies and totals

    bpe_usage[name] = { # store current tokenizer's usage statistics
        "counts": counts, # frequency of individual token IDs by language
        "totals": totals # total token occurrences by language
    }

    print(f"\n{name}") # display current tokenizer name

    for lang in ["en", "tr", "zh"]: # iterate over languages
        print(
            f"  {lang}: {totals[lang]:,} tokens" # display total token occurrences for language
        )


bpe_2000
  en: 1,612,176 tokens
  tr: 1,729,672 tokens
  zh: 2,814,332 tokens

bpe_10000
  en: 884,236 tokens
  tr: 869,649 tokens
  zh: 2,226,238 tokens


In [12]:
# Classify BPE tokens by normalized language association
def classify_vocabulary(tokenizer, usage, threshold=0.80): # define function to classify tokens by language association, using 80% threshold

    counts = usage["counts"] # retrieve token frequency counts by language
    totals = usage["totals"] # retrieve total token occurrences by language

    categories = { # create dictionary containing lists for each classification category
        "en": [],
        "tr": [],
        "zh": [],
        "shared": [],
        "unused": []
    }

    token_details = {} # create dictionary to store classification details for individual tokens

    for token_id in range(tokenizer.get_piece_size()): # iterate over every token ID in tokenizer vocabulary

        # Exclude special tokens from linguistic vocabulary allocation
        if tokenizer.is_control(token_id) or tokenizer.is_unknown(token_id): # identify control and unknown tokens
            continue # skip these special tokens

        # Calculate frequency relative to each language's total tokens
        frequencies = { # calculate normalized token frequency within each language
            lang: counts[lang][token_id] / totals[lang] # divide token occurrences by total tokens in language
            for lang in ["en", "tr", "zh"] # iterate over languages
        }

        frequency_sum = sum(frequencies.values()) # sum normalized frequencies across languages

        # Handle tokens absent from all training data
        if frequency_sum == 0: # check whether token was unused in all three languages
            category = "unused" # classify token as unused
            proportions = {lang: 0.0 for lang in frequencies} # assign zero proportions to all languages

        else:
            # Calculate normalized language proportions
            proportions = { # calculate each language's share of combined normalized frequency
                lang: freq / frequency_sum # divide language frequency by sum of normalized frequencies
                for lang, freq in frequencies.items()
            }

            dominant_lang = max(proportions, key=proportions.get) # identify language with highest normalized proportion

            if proportions[dominant_lang] >= threshold: # check whether dominant language reaches 80% threshold
                category = dominant_lang # classify token as primarily associated with dominant language
            else:
                category = "shared" # classify token as shared if no language reaches threshold

        categories[category].append(token_id) # add token ID to its classification category

        token_details[token_id] = { # store detailed classification information for token
            "piece": tokenizer.id_to_piece(token_id), # retrieve token's readable text representation
            "category": category, # store assigned language category
            "proportions": proportions # store normalized language association proportions
        }

    return categories, token_details # return classified token groups and individual token details

# Apply classification to both BPE vocabularies
allocation_results = {} # create dictionary to store vocabulary allocation results

for name, tokenizer in [ # iterate over both BPE tokenizer names and objects
    ("bpe_2000", bpe_small),
    ("bpe_10000", bpe_large)
]:

    categories, details = classify_vocabulary( # classify current tokenizer's vocabulary
        tokenizer, # supply current BPE tokenizer
        bpe_usage[name] # supply previously calculated token usage statistics
    )

    allocation_results[name] = { # store classification results for current tokenizer
        "categories": categories, # store token IDs grouped by language association
        "details": details # store individual token classification details
    }

    # Count vocabulary entries excluding special tokens
    total_classified = sum( # calculate total number of classified vocabulary entries
        len(token_ids) for token_ids in categories.values() # sum token counts across categories
    )

    print(f"\n{name}: {total_classified:,} vocabulary entries (excluding special tokens)") # display total classified tokens

    for category in ["en", "tr", "zh", "shared", "unused"]: # iterate over classification categories
        count = len(categories[category]) # count vocabulary entries in current category

        print( # display category count and percentage of classified vocabulary
            f"  {category}: {count:,} "
            f"({100 * count / total_classified:.1f}%)"
        )


bpe_2000: 1,997 vocabulary entries (excluding special tokens)
  en: 49 (2.5%)
  tr: 72 (3.6%)
  zh: 1,684 (84.3%)
  shared: 192 (9.6%)
  unused: 0 (0.0%)

bpe_10000: 9,997 vocabulary entries (excluding special tokens)
  en: 1,774 (17.7%)
  tr: 2,205 (22.1%)
  zh: 4,349 (43.5%)
  shared: 1,658 (16.6%)
  unused: 11 (0.1%)


In [13]:
# Display representative tokens from each vocabulary category
for name in ["bpe_2000", "bpe_10000"]: # iterate over both BPE tokenizers

    print(f"\n{name.upper()}") # display tokenizer name in uppercase

    details = allocation_results[name]["details"] # retrieve individual token classification details
    categories = allocation_results[name]["categories"] # retrieve token IDs grouped by language association

    for category in ["en", "tr", "zh", "shared"]: # iterate over four vocabulary categories

        # Sort tokens by their frequency of use
        token_ids = sorted( # sort token IDs by total frequency across all three languages
            categories[category], # retrieve token IDs belonging to current category
            key=lambda token_id: sum( # calculate total occurrences of each token ID
                bpe_usage[name]["counts"][lang][token_id] # retrieve token frequency for current language
                for lang in ["en", "tr", "zh"] # iterate over three languages
            ),
            reverse=True # sort from highest to lowest frequency
        )

        examples = [ # create list of representative token pieces
            details[token_id]["piece"] # retrieve readable text representation of token
            for token_id in token_ids[:10] # select first 10 tokens from sorted list
        ]

        print(f"{category}: {examples}") # display representative tokens for current category


BPE_2000
en: ['▁the', '▁c', '▁of', '▁in', 'ic', '▁to', 'ing', '▁and', '▁w', 'w']
tr: ['ı', '▁k', '▁y', 'ü', 'ın', 'ak', 'ğ', 'z', 'ek', '▁ve']
zh: ['的', '。', '、', '在', '年', '一', '人', '有', '中', '是']
shared: [',', '▁', 'e', '.', 't', 's', 'i', 'm', 'er', 'a']

BPE_10000
en: ['▁the', '▁of', '▁and', 's', '▁in', '▁to', '▁a', '▁is', '▁was', '▁The']
tr: ['▁ve', '▁bir', "▁'", 'lar', 'de', '▁olarak', 'da', 'ın', 'ler', 'ları']
zh: ['。', '的', '▁', '、', '年', '▁,', '在', '(', '和', '人']
shared: [',', '.', "'", ')', '-', 'a', 'in', 'i', '▁(', 'e']


In [14]:
# Save vocabulary allocation results
allocation_summary = {} # create dictionary to store vocabulary allocation counts

for name, result in allocation_results.items(): # iterate over both BPE tokenizers and their classification results

    categories = result["categories"] # retrieve vocabulary categories for current tokenizer

    allocation_summary[name] = { # store category counts under current tokenizer name
        category: len(token_ids) # count vocabulary entries in each category
        for category, token_ids in categories.items() # iterate over categories and their token ID lists
    }

with open(MODEL_DIR / "vocabulary_allocation.json", "w") as f: # open JSON file for writing
    json.dump(allocation_summary, f, indent=2) # save vocabulary allocation counts in readable JSON format

print("Vocabulary allocation results saved.") # confirm results were saved

Vocabulary allocation results saved.


### 5. Sentence-Level Error Analysis

In [15]:
# Display candidate test sentences for error analysis
for lang in ["en", "tr", "zh"]: # iterate over English, Turkish, and Chinese

    print(f"\n{lang.upper()} TEST SENTENCES") # display current language heading in uppercase

    # Select sentences of moderate length
    candidates = [ # create list of sentences meeting length requirements
        sentence # select current sentence
        for sentence in test_texts[lang] # iterate over current language's test sentences
        if 60 <= len(sentence) <= 120 # include sentences containing 60 to 120 Unicode characters
    ]

    # Display first five candidates
    for i, sentence in enumerate(candidates[:5], 1): # enumerate first five candidates, numbering from 1
        print(f"{i}. {sentence}") # display numbered candidate sentence


EN TEST SENTENCES
1. Siegfried kills Mime by chopping his head off with the sword.
2. Oklahoma City is the state's capital and largest city by population and land area.
3. The troubles stemmed from the cultural clash between the tribal groups and the plains people.
4. 'Archived copy' and 'Archive copy' are commonly provided by bots that are unable to identify the source's correct title.
5. Most of a dragonfly's life is spent in the nymph form, beneath the water's surface.

TR TEST SENTENCES
1. Rivayete göre kırklar mevkiinden geçen gelin buradaki kavak ağaçlarının Tanrıya secde ettiklerini görür.
2. Bundan dolayı alternatif diyotlar verimli güç dönüşümü için kullanılmak zorundadırlar.
3. Dünya Gümrük Örgütü bünyesinde imzalanan bazı uluslararası sözleşmelere Türkiye de taraf olmuştur.
4. Lamdre’nin (yol ve meyve) Sakya sistemi, Hevajra- Tantra metinleri ile tanınmıştır.
5. Ayrıca plastik kasasından ötürü mayın tarama aygıtları ile bulunması çok güçtür.

ZH TEST SENTENCES
1. 广场公园共分為始绿园

#### 5.1 Sentence Tokenization and Model Scores

In [16]:
# Select three test sentences for error analysis
selected_sentences = { # create dictionary containing one selected sentence per language
    "en": [s for s in test_texts["en"] if "dragonfly's life" in s][0], # select English sentence containing specified phrase
    "tr": [s for s in test_texts["tr"] if "ağaçlarının" in s][0], # select Turkish sentence containing specified word
    "zh": [s for s in test_texts["zh"] if "广场公园共分" in s][0] # select Chinese sentence containing specified characters
}

# Evaluate one sentence using a trained model
def evaluate_sentence(sentence, tokenizer_name): # define function to evaluate sentence using selected tokenizer/model

    # Encode the sentence
    encoded = encode_sentences( # convert sentence into token IDs
        {"sample": [sentence]}, # create dictionary containing sentence under sample key
        tokenizer_name # specify tokenizer to use
    )

    tokens = encoded["sample"][0] # retrieve encoded token ID sequence

    # Use the same sequence construction as training
    dataset = LanguageModelDataset( # create dataset object for selected sentence
        encoded, # use encoded sentence
        context_length=MODEL_CONFIG["context_length"] # use original model context length
    )

    loader = DataLoader( # create DataLoader for sentence evaluation
        dataset, # use prepared sentence dataset
        batch_size=1, # process one training example at a time
        shuffle=False, # preserve sequence order
        collate_fn=collate_lm_batch # use established padding function
    )

    model = models[tokenizer_name] # retrieve corresponding trained Transformer
    model.eval() # switch model to evaluation mode

    total_loss = 0.0 # initialize accumulated sentence prediction loss

    with torch.no_grad(): # disable gradient calculations during evaluation

        for inputs, targets in loader: # iterate over sentence evaluation batches

            inputs = inputs.to(device) # move inputs to selected computing device
            targets = targets.to(device) # move targets to selected computing device

            logits = model(inputs) # generate prediction scores for sentence tokens

            loss = F.cross_entropy( # calculate sentence prediction error
                logits.reshape(-1, logits.size(-1)), # reshape prediction scores for loss calculation
                targets.reshape(-1), # flatten target token IDs
                ignore_index=-100, # exclude padded positions
                reduction="sum" # sum prediction losses across tokens
            )

            total_loss += loss.item() # accumulate total sentence prediction loss

    # Normalize loss by original sentence length
    bpc = total_loss / (math.log(2) * len(sentence)) # calculate bits per original Unicode character

    return tokens, bpc # return encoded token IDs and sentence BPC

# Compare tokenizations and scores
sentence_results = {} # create dictionary to store sentence-level evaluation results

for lang, sentence in selected_sentences.items(): # iterate over selected sentences by language

    print(f"\n{'=' * 60}") # print separator
    print(f"LANGUAGE: {lang.upper()}") # display language
    print(f"SENTENCE: {sentence}") # display original sentence
    print(f"{'=' * 60}") # print separator

    sentence_results[lang] = {} # create dictionary for current language's results

    for name in ["character", "bpe_2000", "bpe_10000"]: # evaluate sentence using each model

        tokens, bpc = evaluate_sentence(sentence, name) # retrieve token IDs and BPC

        # Convert token IDs into readable token pieces
        if name == "character": # character-level tokenizer
            pieces = list(sentence) # represent sentence as individual characters

        elif name == "bpe_2000": # small BPE tokenizer
            pieces = [
                bpe_small.id_to_piece(token) # convert token ID to readable BPE piece
                for token in tokens # iterate over encoded token IDs
            ]

        else: # large BPE tokenizer
            pieces = [
                bpe_large.id_to_piece(token) # convert token ID to readable BPE piece
                for token in tokens # iterate over encoded token IDs
            ]

        sentence_results[lang][name] = { # store results for current language and model
            "token_count": len(tokens), # number of tokens representing sentence
            "bpc": bpc, # sentence-level bits per character
            "pieces": pieces # readable tokenization
        }

        print(f"\n{name}") # display model name
        print(f"Tokens: {len(tokens)}") # display token count
        print(f"BPC: {bpc:.4f}") # display sentence BPC
        print(f"Tokenization: {pieces}") # display token pieces


LANGUAGE: EN
SENTENCE: Most of a dragonfly's life is spent in the nymph form, beneath the water's surface.

character
Tokens: 83
BPC: 2.5776
Tokenization: ['M', 'o', 's', 't', ' ', 'o', 'f', ' ', 'a', ' ', 'd', 'r', 'a', 'g', 'o', 'n', 'f', 'l', 'y', "'", 's', ' ', 'l', 'i', 'f', 'e', ' ', 'i', 's', ' ', 's', 'p', 'e', 'n', 't', ' ', 'i', 'n', ' ', 't', 'h', 'e', ' ', 'n', 'y', 'm', 'p', 'h', ' ', 'f', 'o', 'r', 'm', ',', ' ', 'b', 'e', 'n', 'e', 'a', 't', 'h', ' ', 't', 'h', 'e', ' ', 'w', 'a', 't', 'e', 'r', "'", 's', ' ', 's', 'u', 'r', 'f', 'a', 'c', 'e', '.']

bpe_2000
Tokens: 47
BPC: 2.6256
Tokenization: ['▁M', 'o', 'st', '▁of', '▁a', '▁d', 'ra', 'g', 'on', 'f', 'ly', "'", 's', '▁l', 'if', 'e', '▁is', '▁s', 'p', 'ent', '▁in', '▁the', '▁n', 'y', 'm', 'p', 'h', '▁for', 'm', ',', '▁b', 'en', 'e', 'at', 'h', '▁the', '▁w', 'at', 'er', "'", 's', '▁s', 'ur', 'f', 'ac', 'e', '.']

bpe_10000
Tokens: 31
BPC: 2.3482
Tokenization: ['▁Most', '▁of', '▁a', '▁d', 'rag', 'on', 'f', 'ly', "'", 's

In [17]:
# Save sentence-level error analysis results
with open(MODEL_DIR / "sentence_analysis.json", "w", encoding="utf-8") as f: # open JSON file in write mode using UTF-8 encoding
    json.dump( # save sentence-level analysis results to JSON
        {
            "selected_sentences": selected_sentences, # store original selected sentences
            "results": sentence_results # store token counts, BPC scores, and tokenizations
        },
        f, # specify destination file
        indent=2, # format JSON with readable indentation
        ensure_ascii=False # preserve original Unicode characters in saved text
    )

print("Sentence-level analysis saved successfully.") # confirm results were saved

Sentence-level analysis saved successfully.
